In [2]:
# Importing all variables from .env for mongodb
from langchain_mongodb import MongoDBAtlasVectorSearch
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
MONGO_USER = os.getenv("MONGO_USER")
MONGO_PASSWORD = os.getenv("MONGO_PASSWORD")
MONGO_CLUSTER = os.getenv("MONGO_CLUSTER")
MONGO_DATABASE = os.getenv("MONGO_DATABASE")

uri = f"mongodb+srv://{MONGO_USER}:{MONGO_PASSWORD}@{MONGO_CLUSTER}.igv4jfg.mongodb.net/?retryWrites=true&w=majority&appName={MONGO_CLUSTER}"

client = MongoClient(uri, server_api=ServerApi('1'))
db = client.get_database(MONGO_DATABASE)

rev_embeddings_collection = db["review_embeddings"]

In [ ]:
# Data Filtering and converting into clean dictionary, later for embedding
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from groq import Groq

loader = TextLoader("docs/fake_internships_info.md")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
split_docs = splitter.split_text(docs[0].page_content)

embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

data = []
for chunk in split_docs:
    reviews = {
        "text": chunk,
        "embeddings": embeddings.embed_query(chunk)
    }
    data.append(reviews)

rev_embeddings_collection.insert_many(data) # This is meant to insert data into mongodb

In [11]:
# Vector Store retrieval
vector_store = MongoDBAtlasVectorSearch(
    collection=rev_embeddings_collection,
    embedding=embeddings,
    index_name="REVIEW_VECTOR_INDEX",
    relevance_score_fn="cosine",
    embedding_key="embeddings"
)
def ask(query: str):
    retrieved_docs = vector_store.similarity_search(query=query, k=3)
    context = '\n\n'.join([doc.page_content for doc in retrieved_docs])
    prompt = f"""
    You are a helpful Assistant who helps people to:
    - Helps people to guide about their career
    - Helps people to detect red flags in an internship, copy pasted from description or told in a question
    - Helps people to suggest advice related to finding internships

    Rules:
    - If someone asks query about detecting red flags or query related to joining in an internship, Use the following context provided from the knowledge base and apply:
    {context}

    - If someone asks query about guidance / mentorship related to career, your job is to guide them in a warm soft manner like a friend
    Remember you are a hope for someone who is desperate and need to advance his career or get jobs.

    The question is given below:
    {query}
    """
    client = Groq(api_key=GROQ_API_KEY)
    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content" : prompt},
            {"role": "user", "content": query}
        ],
        temperature=0.5,
        model="llama-3.1-8b-instant"
        )
    return response

res = ask("Hey I recently learned Wordpress and applied for internship as Web Dev internee at CodSoft on Linkedin, but they are asking for fees, they promise that they will mentor me on processing fee request of nearly 1000 Rs")
print(res.choices[0].message.content)

I'm glad you reached out to me about this. It sounds like you're excited about your new skills in WordPress and eager to gain experience through an internship.

Firstly, congratulations on taking the first step by applying for an internship on LinkedIn. That's a great way to start building your professional network.

Now, let's talk about the red flag you've mentioned. CodSoft is one of the companies that have been reported to be involved in fake internship scams. They're asking for a processing fee of nearly 1000 Rs, which is a huge red flag.

Here's what you should know: legitimate internships don't charge fees. Period. If a company is asking you to pay a fee for an internship, it's likely a scam. They might promise you mentorship, but in reality, you'll be doing tasks without any guidance or support.

I would strongly advise you to be cautious and not pay the fee. Instead, you could try to reach out to CodSoft and ask them some questions like:

* What kind of tasks will I be doing a